<a href="https://colab.research.google.com/github/Tejal-singhh/OIBSIP/blob/main/DataAnalytics-Level1-DataCleaning/Tejalsingh_Task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fifa21 raw data v2.csv to fifa21 raw data v2.csv


In [ ]:
import pandas as pd

df = pd.read_csv("fifa21 raw data v2.csv", low_memory=False)
df.shape

(18979, 77)

In [ ]:
cols = ['ID', 'Name', 'Age', 'Nationality', 'Club', 'Contract', 'Height', 'Weight',
        'Preferred Foot', 'Positions', 'Value', 'Wage', 'Release Clause',
        'Joined', 'Loan Date End', 'Hits', 'W/F', 'SM', 'IR']

df = df[cols].copy()
df.head()

,ID,Name,Age,Nationality,Club,Contract,Height,Weight,Preferred Foot,Positions,Value,Wage,Release Clause,Joined,Loan Date End,Hits,W/F,SM,IR
0,158023,L. Messi,33,Argentina,\n\n\n\nFC Barcelona,2004 ~ 2021,170cm,72kg,Left,"RW, ST, CF",€103.5M,€560K,€138.4M,"Jul 1, 2004",NaN,771,4 ★,4★,5 ★
1,20801,Cristiano Ronaldo,35,Portugal,\n\n\n\nJuventus,2018 ~ 2022,187cm,83kg,Right,"ST, LW",€63M,€220K,€75.9M,"Jul 10, 2018",NaN,562,4 ★,5★,5 ★
2,200389,J. Oblak,27,Slovenia,\n\n\n\nAtlético Madrid,2014 ~ 2023,188cm,87kg,Right,GK,€120M,€125K,€159.4M,"Jul 16, 2014",NaN,150,3 ★,1★,3 ★
3,192985,K. De Bruyne,29,Belgium,\n\n\n\nManchester City,2015 ~ 2023,181cm,70kg,Right,"CAM, CM",€129M,€370K,€161M,"Aug 30, 2015",NaN,207,5 ★,4★,4 ★
4,190871,Neymar Jr,28,Brazil,\n\n\n\nParis Saint-Germain,2017 ~ 2022,175cm,68kg,Right,"LW, CAM",€132M,€270K,€166.5M,"Aug 3, 2017",NaN,595,5 ★,5★,5 ★


In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nNull counts:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape: (18979, 19)

Data types:
 ID                 int64
Name              object
Age                int64
Nationality       object
Club              object
Contract          object
Height            object
Weight            object
Preferred Foot    object
Positions         object
Value             object
Wage              object
Release Clause    object
Joined            object
Loan Date End     object
Hits              object
W/F               object
SM                object
IR                object
dtype: object

Null counts:
 ID                    0
Name                  0
Age                   0
Nationality           0
Club                  0
Contract              0
Height                0
Weight                0
Preferred Foot        0
Positions             0
Value                 0
Wage                  0
Release Clause        0
Joined                0
Loan Date End     17966
Hits               2595
W/F                   0
SM                    0
IR                    0
dtype: 

 Data Quality Report

- Shape: 18,979 rows × 19 columns
- Duplicates: 0 duplicate rows found
- Nulls: `Loan Date End` has ~17,966 missing values (expected — most players aren't on loan), `Hits` has ~2,595 missing values (needs handling)
- Data type issues: `Value`, `Wage`, `Release Clause`, `Height`, `Weight`, `Joined` are all stored as text (object) instead of proper numeric/date types

In [ ]:
df['Age'].describe()

,Age
count,18979.000000
mean,25.194109
std,4.710520
min,16.000000
25%,21.000000
50%,25.000000
75%,29.000000
max,53.000000


 Value range check: `Age` ranges from 16 to 53 — both realistic, no invalid entries

In [ ]:
df['Loan Date End'] = df['Loan Date End'].fillna('Not on Loan')
df['Hits'] = df['Hits'].fillna(0)

df[['Loan Date End', 'Hits']].isnull().sum()

,0
Loan Date End,0
Hits,0


Missing Data Handling

- Loan Date End: Missing values mean the player is not currently on loan — filled with `'Not on Loan'` instead of imputing a fake date, since there's no numeric/date value that would make sense here.
- Hits: Missing values filled with `0`, assuming no recorded value means no hits were tracked for that player, rather than data being lost. No rows were deleted, since both columns' missingness is meaningful rather than random.

In [ ]:
before_rows = df.shape[0]
df = df.drop_duplicates()
after_rows = df.shape[0]

print("Rows removed:", before_rows - after_rows)

Rows removed: 0


Duplicate Removal

Checked for fully duplicate rows using `drop_duplicates()`. 0 duplicate rows found — each player record is unique, likely because `ID` is a unique player identifier. No rows were removed at this step.

In [ ]:
import re

def convert_height(h):
    h = str(h)
    if 'cm' in h:
        return int(h.replace('cm', ''))
    else:
        # format like 6'2"
        feet, inches = re.findall(r'\d+', h)
        return round((int(feet) * 12 + int(inches)) * 2.54)

def convert_weight(w):
    w = str(w)
    if 'kg' in w:
        return int(w.replace('kg', ''))
    else:
        lbs = int(w.replace('lbs', ''))
        return round(lbs * 0.453592)

df['Height'] = df['Height'].apply(convert_height)
df['Weight'] = df['Weight'].apply(convert_weight)

df[['Height', 'Weight']].describe()

,Height,Weight
count,18979.000000,18979.000000
mean,181.200221,75.019021
std,6.840054,7.073542
min,155.000000,50.000000
25%,176.000000,70.000000
50%,181.000000,75.000000
75%,186.000000,80.000000
max,206.000000,110.000000


Standardization — Height & Weight

Both columns originally mixed two different unit systems: `Height` mixed metric (`"170cm"`) with imperial (`"6'2\""`), and `Weight` mixed kg with lbs. Wrote conversion functions to detect the format per value and standardize everything into cm and kg.
Result: both columns are now clean numeric types with realistic, consistent ranges (Height: 155-206cm, Weight: 50-110kg).

In [ ]:
def convert_money(val):
    val = str(val).replace('€', '')
    if 'M' in val:
        return float(val.replace('M', '')) * 1_000_000
    elif 'K' in val:
        return float(val.replace('K', '')) * 1_000
    else:
        return float(val)

df['Value'] = df['Value'].apply(convert_money)
df['Wage'] = df['Wage'].apply(convert_money)
df['Release Clause'] = df['Release Clause'].apply(convert_money)

df[['Value', 'Wage', 'Release Clause']].describe()

,Value,Wage,Release Clause
count,1.897900e+04,18979.000000,1.897900e+04
mean,2.865063e+06,9092.062279,3.962951e+06
std,7.685154e+06,19707.021089,9.772762e+06
min,0.000000e+00,0.000000,0.000000e+00
25%,4.750000e+05,1000.000000,4.235000e+05
50%,9.500000e+05,3000.000000,1.000000e+06
75%,2.000000e+06,8000.000000,2.800000e+06
max,1.855000e+08,560000.000000,2.031000e+08


Standardization — Value, Wage, Release Clause

All three columns were stored as text with a `€` symbol and shorthand suffixes (`M` for million, `K` for thousand), e.g. `"€103.5M"`. Wrote a function to strip the currency symbol and convert the shorthand into actual numeric values. All three are now proper float columns in raw Euros, ready for numeric analysis.

In [ ]:
df['Joined'] = pd.to_datetime(df['Joined'])
df['Joined'].head()

,Joined
0,2004-07-01
1,2018-07-10
2,2014-07-16
3,2015-08-30
4,2017-08-03


Standardization — Joined Date

Converted `Joined` from text (e.g. `"Jul 1, 2004"`) into a proper datetime type using `pd.to_datetime()`, enabling date-based calculations like tenure length or sorting by join date.

In [ ]:
for col in ['W/F', 'SM', 'IR']:
    df[col] = df[col].str.replace('★', '', regex=False).str.strip().astype(int)

df[['W/F', 'SM', 'IR']].head()

,W/F,SM,IR
0,4,4,5
1,4,5,5
2,3,1,3
3,5,4,4
4,5,5,5


Standardization — Star Rating Columns (W/F, SM, IR)

These columns had inconsistent formatting — e.g. `"4 ★"` vs `"4★"` (differing spacing around the star symbol). Removed the star character and stripped whitespace, converting all three columns into clean integers.

In [ ]:
def convert_hits(val):
    val = str(val)
    if 'K' in val:
        return float(val.replace('K', '')) * 1000
    else:
        return float(val)

df['Hits'] = df['Hits'].apply(convert_hits)
df['Hits'].describe()

,Hits
count,18979.000000
mean,22.936720
std,119.861937
min,0.000000
25%,1.000000
50%,4.000000
75%,12.000000
max,8400.000000


 Standardization — Hits

`Hits` mixed plain numbers with shorthand values like `"1.6K"`. Converted all shorthand values by multiplying by 1000, resulting in a clean numeric column. Note the strong right-skew (median 4, max 8,400) — likely genuine, since a handful of globally famous players would naturally get far more profile views than average.

In [ ]:
def find_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

for col in ['Value', 'Wage', 'Release Clause', 'Hits']:
    count, lower, upper = find_outliers_iqr(col)
    print(f"{col}: {count} outliers | valid range: {lower:.2f} to {upper:.2f}")

Value: 2297 outliers | valid range: -1812500.00 to 4287500.00
Wage: 2356 outliers | valid range: -9500.00 to 18500.00
Release Clause: 2851 outliers | valid range: -3141250.00 to 6364750.00
Hits: 2559 outliers | valid range: -15.50 to 28.50


 Outlier Detection (IQR Method)

Checked `Value`, `Wage`, `Release Clause`, and `Hits` using the IQR method (1.5×IQR rule). All four columns show a large number of statistical outliers (2,297–2,851 rows), but this reflects genuine real-world skew, not data errors — a small number of globally famous players (e.g. Messi, Ronaldo) legitimately have far higher value, wage, and hits than typical players.

Decision: Retain all outliers. Removing or capping them would delete meaningful information about star players and distort the dataset's real-world accuracy. These are legitimate extreme values, not anomalies.

In [ ]:
df['ID'] = df['ID'].astype(str)

df.dtypes

,0
ID,object
Name,object
Age,int64
Nationality,object
Club,object
Contract,object
Height,int64
Weight,int64
Preferred Foot,object
Positions,object


## Data Type Correction

Converted `ID` from integer to string, since it's an identifier/label, not a quantity to calculate with. All other columns now hold correct types: numeric columns (Age, Height, Weight, Value, Wage, Release Clause, Hits, W/F, SM, IR) are int64/float64, dates are datetime64, and text/categorical fields remain as strings.

In [ ]:
df_original = pd.read_csv("fifa21 raw data v2.csv", low_memory=False)[cols]

summary = pd.DataFrame({
    'Metric': ['Row Count', 'Null Count', 'Duplicate Count', 'Correct Dtypes'],
    'Before Cleaning': [
        df_original.shape[0],
        df_original.isnull().sum().sum(),
        df_original.duplicated().sum(),
        (df_original.dtypes != 'object').sum()
    ],
    'After Cleaning': [
        df.shape[0],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        (df.dtypes != 'object').sum()
    ]
})
summary

,Metric,Before Cleaning,After Cleaning
0,Row Count,18979,18979
1,Null Count,20561,0
2,Duplicate Count,0,0
3,Correct Dtypes,2,11


## Before vs. After Summary

| Metric | Before Cleaning | After Cleaning |
|---|---|---|
| Row Count | 18,979 | 18,979 |
| Null Count | 20,561 | 0 |
| Duplicate Count | 0 | 0 |
| Correctly-Typed Columns | 2 | 11 |

No rows were lost during cleaning — all missing values were meaningfully filled rather than dropped, and outliers were retained as genuine data. The dataset went from having 20,561 nulls and mostly text-typed columns, to a fully null-free dataset with correct data types across numeric, date, and categorical fields.

In [ ]:
df.to_csv("fifa21_cleaned.csv", index=False)

from google.colab import files
files.download("fifa21_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

This dataset was cleaned systematically across 5 dimensions: missing values, duplicates, formatting inconsistencies, outliers, and data types.

**Key decisions:**
1. **Missing values were filled meaningfully, not blindly** — `Loan Date End` nulls represent "not on loan" (a real status, not missing info), and `Hits` nulls were treated as 0 recorded views, so no rows were deleted.
2. **Standardization required custom logic** — Height/Weight mixed metric and imperial units, and Value/Wage/Release Clause used currency shorthand (`M`/`K`). Both needed conditional conversion functions rather than a simple find-and-replace.
3. **Outliers were retained, not removed** — the extreme values in Value, Wage, and Hits reflect real-world superstar players, not data entry errors. Removing them would have made the dataset less representative of reality, not more accurate.

**Result:** Zero nulls, zero duplicates, all 19 columns in their correct data type, and no rows lost — the dataset is now fully analysis-ready while preserving 100% of the original information.